In [ ]:
from bgm_archive.graph import BgmGraph

DB_PATH = ":memory:"
DB_PATH = "try.duckdb"

bg = BgmGraph(path=DB_PATH, read_only=False)

In [ ]:
bg.setup_db()

In [ ]:
extensions = bg.list_extensions().df()
extensions

In [ ]:
bg._conn.execute("""
                 CREATE TABLE Person AS SELECT * FROM 'https://gist.githubusercontent.com/Dtenwolde/2b02aebbed3c9638a06fda8ee0088a36/raw/8c4dc551f7344b12eaff2d1438c9da08649d00ec/person-sf0.003.csv';
CREATE TABLE Person_knows_person AS SELECT * FROM 'https://gist.githubusercontent.com/Dtenwolde/81c32c9002d4059c2c3073dbca155275/raw/8b440e810a48dcaa08c07086e493ec0e2ec6b3cb/person_knows_person-sf0.003.csv';
                 """).df()

In [ ]:
bg._conn.table("Person").df()

In [ ]:
bg._conn.table("Person_knows_person").df()

In [ ]:
bg._conn.execute("""
                 CREATE PROPERTY GRAPH snb
VERTEX TABLES (
    Person
  )
EDGE TABLES (
    Person_knows_person 
        SOURCE KEY ( person1id ) REFERENCES Person ( id )
        DESTINATION KEY ( person2id ) REFERENCES Person ( id )
        LABEL Knows
  ); """).df()

In [ ]:
bg._conn.execute("SHOW TABLES").df()

In [ ]:
bg._conn.execute("""FROM GRAPH_TABLE(snb
    MATCH (a:Person WHERE a.firstName = 'Jan')-[k:Knows]->(b:Person)
    COLUMNS (b.firstName)
); 
""").df()